# Prompt Evaluation Workflow

1. **Draft a Prompt**
   - Write the initial prompt and define the expected output.

2. **Create an Evaluation Dataset**
   - Prepare test cases with sample inputs and expected outputs.

3. **Feed Through Claude**
   - Run the prompt through Claude using the evaluation dataset.
   - Collect and record the generated outputs.

4. **Feed Through Grader**
   - Evaluate Claude's outputs against predefined criteria.
   - Measure accuracy, relevance, consistency, and instruction adherence.

5. **Change Prompt and Repeat**
   - Analyze evaluation results and identify weaknesses.
   - Refine the prompt based on the grader's feedback.
   - Repeat the evaluation workflow until the desired performance is achieved.

In [1]:
# load env variables from .env file
from dotenv import load_dotenv
load_dotenv()
import os

In [2]:
# create api client 
from anthropic import Anthropic
client = Anthropic(
    base_url = os.getenv("ANTHROPIC_BASE_URL"),
    api_key = os.getenv("ANTHROPIC_AUTH_TOKEN"),
)

model_name = os.getenv("ANTHROPIC_MODEL")

In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model_name,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [4]:
import json

import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for prompts that generate Python, JSON, or Regex
for AWS-related tasks.

Requirements:
- Return a JSON array containing exactly 3 objects.
- Each object must contain a "task" field describing an AWS-related task.
- Each task must be solvable using a single Python function, a single JSON object,
  or a single regex.
- Keep tasks short and simple.
- Return only valid JSON without Markdown code fences.

Example:
[
    {"task": "Write a Python function to validate an S3 bucket name."},
    {"task": "Create a JSON object representing an AWS Lambda configuration."},
    {"task": "Write a regex to match an AWS ARN."}
]
"""

    messages = []
    add_user_message(messages, prompt)

    text = chat(messages)

    dataset = json.loads(text)

    if len(dataset) != 3:
        raise ValueError(f"Expected 3 tasks, but received {len(dataset)}")

    return dataset

In [5]:
dataset = generate_dataset()

with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)

In [6]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [7]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = """
    You are an expert code reviewer. Evaluate this AI-generated solution.
    
    Task: {task}
    Solution: {solution}
    
    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [8]:
def run_test_case(test_case):
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output, 
        "test_case": test_case, 
        "score": score,
        "reasoning": reasoning
    }

In [9]:
from statistics import mean

def run_eval(dataset):
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [10]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 6.666666666666667


In [11]:
print(json.dumps(results, indent=2))

[
  {
    "output": "To validate an S3 bucket name using a regular expression (regex), we need to follow the AWS guidelines for bucket names. According to AWS, bucket names must:\n\n1. Be 1 to 63 characters long.\n2. Contain only lowercase letters, numbers, and hyphens.\n3. Start and end with a letter or a number.\n4. Not contain consecutive hyphens.\n5. Not start or end with a hyphen.\n\nBased on these rules, here's a regex pattern to validate an S3 bucket name:\n\n```regex\n^[a-z]([-a-z0-9]{0,61}[a-z0-9])$\n```\n\n### Explanation:\n- `^` asserts the start of the string.\n- `[a-z]` ensures the bucket name starts with a letter.\n- `([-a-z0-9]{0,61}[a-z0-9])` matches any combination of letters, numbers, or hyphens, with exactly one occurrence of the letter or number at the end. The `{0,61}` ensures up to 61 characters in total.\n- `$` asserts the end of the string.\n\nLet's test this regex with a few examples:\n\n- Valid examples: `test`, `test123`, `1test`, `-test`\n- Invalid examples: